# Fucus Dispersal — Heatmaps

In [ ]:
import numpy as np
import xarray as xr
from xhistogram.xarray import histogram as xhist

import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import cmocean

import cartopy.crs as ccrs
from cartopy.crs import PlateCarree

from pathlib import Path

# Parameters

In [ ]:
# Parameters
base_path = "/gxfs_work/geomar/smomw122/2025_fucus-dispersal"

# Which experiment to plot: "surface", "bottom", or "surface_stokes"
experiment_type = "surface"

output_dt_mins = 60

# Heatmap spatial bins
n_bins = 120
lon_min, lon_max = 5, 32
lat_min, lat_max = 53, 66

# Heatmap temporal binning (days since release)
day_bin_width = 10

In [ ]:
base_path = Path(base_path)

trajectory_path = base_path / "output" / "Trajectories" / experiment_type
zarr_files = sorted(trajectory_path.glob("**/*.zarr"))
print(f"{len(zarr_files)} trajectory files found for experiment_type={experiment_type}")

ds = xr.concat([xr.open_zarr(z) for z in zarr_files], dim="trajectory")
print(f"{ds.sizes['trajectory']} trajectories, {ds.sizes['obs']} obs steps")
print(f"{ds.nbytes / 1e9:.1f} GB")
ds

# BSH model coastline

Load staircase coastline polygons from GeoJSON (produced by
`scripts/004_extract_coastline.py`).

In [ ]:
import geopandas as gpd

coastline_path = base_path / "data" / "BSH_model_coastline" / "coastline.geojson"
gdf_coastline = gpd.read_file(coastline_path)
print(gdf_coastline[["grid"]].value_counts())
gdf_coastline

# Heatmaps every 10 days

In [ ]:
lon_bins = np.linspace(lon_min, lon_max, n_bins)
lat_bins = np.linspace(lat_min, lat_max, n_bins)

# obs index to days since release
obs_per_day = 24 * 60 // output_dt_mins
n_obs = ds.sizes["obs"]
max_days = n_obs // obs_per_day

day_edges = np.arange(0, max_days + day_bin_width, day_bin_width)
print(f"obs_per_day={obs_per_day}, max_days={max_days}, bins: {day_edges}")

In [ ]:
heatmaps = []
for d0, d1 in zip(day_edges[:-1], day_edges[1:]):
    obs_start = d0 * obs_per_day
    obs_end = min(d1 * obs_per_day, n_obs)
    ds_slice = ds.isel(obs=slice(obs_start, obs_end))
    h = xhist(
        ds_slice.lon, ds_slice.lat,
        bins=[lon_bins, lat_bins],
        dim=["trajectory", "obs"],
    ).rename(lon_bin="lon", lat_bin="lat")
    h = h.assign_coords(day_start=d0, day_end=d1)
    heatmaps.append(h)
    print(f"  days {d0:3d}–{d1:3d}: obs {obs_start}–{obs_end}, max count = {int(h.max())}")

ds_heat = xr.concat(heatmaps, dim="period")
ds_heat

In [ ]:
n_periods = ds_heat.sizes["period"]
ncols = min(n_periods, 4)
nrows = int(np.ceil(n_periods / ncols))

extent = (lon_min, lon_max, lat_min, lat_max)
projection = ccrs.EckertIV(central_longitude=np.mean([lon_min, lon_max]))

vmax = float(ds_heat.where(ds_heat > 0).quantile(0.95))
norm = LogNorm(vmin=1, vmax=vmax)

fig, axes = plt.subplots(
    nrows=nrows, ncols=ncols,
    figsize=(5 * ncols, 5 * nrows),
    subplot_kw=dict(projection=projection),
    squeeze=False,
)

for idx in range(nrows * ncols):
    ax = axes.flat[idx]
    if idx >= n_periods:
        ax.set_visible(False)
        continue

    h = ds_heat.isel(period=idx)
    d0, d1 = int(h.day_start), int(h.day_end)

    im = h.plot.pcolormesh(
        x="lon", y="lat", ax=ax,
        cmap=cmocean.cm.thermal,
        norm=norm,
        add_colorbar=False,
        transform=PlateCarree(),
    )

    # Model coastline
    gdf_coastline.boundary.plot(
        ax=ax, transform=PlateCarree(),
        color="0.4", linewidth=0.3,
    )

    ax.set_title(f"Days {d0}–{d1}")
    ax.set_extent(extent)
    ax.gridlines(draw_labels=False, linewidth=0.3)

fig.colorbar(im, ax=axes, label="Particle count", shrink=0.6)
fig.suptitle(f"Particle heatmaps ({len(zarr_files)} files, {day_bin_width}-day bins)", y=1.01)
fig.tight_layout()
plt.show()